# 02 — Storage format and layout experiments

BFT65B · Student ID 11230945

Measure uncompressed CSV against flat and year/quarter partitioned Parquet using Snappy and ZSTD. Verify equality before timing. The source is the sealed acquisition from Notebook 01.

## 1. Repository and pinned runtime

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    actual = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    if actual != REPO_URL:
        raise RuntimeError(f"Unexpected repository: {actual}")
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
# The research commands run in fresh Python processes to avoid stale imports.
def command(*args):
    subprocess.run([sys.executable, str(REPO / "run_part1.py"), *map(str, args)], cwd=REPO, check=True)


## 2. Mandatory offline validation

In [ ]:
import tempfile
import xml.etree.ElementTree as ET

TESTS_PASSED = False
with tempfile.TemporaryDirectory() as tmp:
    result_path = Path(tmp) / "tests.xml"
    env = dict(os.environ, PYTEST_DISABLE_PLUGIN_AUTOLOAD="1")
    # Stream the complete failure traceback instead of truncating captured output.
    subprocess.run([sys.executable, "-m", "pytest", "-q", "--tb=short",
                    "--show-capture=no", f"--junitxml={result_path}"],
                   cwd=REPO, env=env, check=True)
    suites = ET.parse(result_path).findall(".//testsuite")
    total = sum(int(s.get("tests", 0)) for s in suites)
    skipped = sum(int(s.get("skipped", 0)) for s in suites)
    if total == 0 or skipped:
        raise RuntimeError(f"Incomplete tests: total={total}, skipped={skipped}")
    TESTS_PASSED = True
    print(f"Mandatory offline validation: PASS ({total} tests, zero skips)")


## 3. Resolve the verified source

In [ ]:
from google.colab import drive
import json

drive.mount("/content/drive")
WORKSPACE = Path("/content/drive/MyDrive/11230945")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKSPACE)


In [ ]:
assert TESTS_PASSED
assert not (WORKSPACE / "pending_source.json").exists(), "Complete Notebook 01 first"
command("verify", "--workspace", WORKSPACE)
source_pointer = json.loads((WORKSPACE / "active_source.json").read_text())
print("Source manifest SHA-256:", source_pointer["manifest_sha256"])


## 4. Experimental protocol

| Factor | Levels |
|---|---|
| Storage representation | CSV; flat Snappy/ZSTD; partitioned Snappy/ZSTD |
| Workload | Full scan; six-column projection; 2020; 2020Q1 |
| Codec | Snappy default; ZSTD level 3 |
| Repetition | 1 warm-up and 5 measured runs for each pair |
| Supplemental study | pandas versus Polars, same four-column CSV projection |

All files are staged on Colab local disk. The worker reads the same logical columns, checks equivalent outputs, and records fresh-process RSS. Partition filters include explicit year/quarter predicates; candidate fragments are not physical bytes read. OS caching is not controlled. Imports and process startup are excluded from query timing. Preparation, full equality comparisons and plots are not timed.

In [ ]:
import shutil

LOCAL = Path("/content/fdic_benchmark_scratch")
LOCAL.mkdir(exist_ok=True)
print("Free local disk (GiB):", round(shutil.disk_usage(LOCAL).free / 2**30, 2))
if shutil.disk_usage(LOCAL).free < 5 * 2**30:
    raise RuntimeError("Insufficient scratch disk; free at least 5 GiB before benchmarking")
WARMUPS, REPETITIONS, THREADS = 1, 5, 2
print("Expected worker executions:", (5*4+2)*(WARMUPS+REPETITIONS))


## 5. Prepare formats, check parity and run benchmark
The command stages the representations, compares values and null masks for every workload, runs the randomized measurement blocks and copies the evidence to Drive only after all checks pass. A failed run does not publish a new evidence pointer.

In [ ]:
assert TESTS_PASSED
BENCHMARK_COMPLETED = False
command("benchmark", "--workspace", WORKSPACE, "--local", LOCAL,
        "--warmups", WARMUPS, "--repetitions", REPETITIONS, "--threads", THREADS)
BENCHMARK_COMPLETED = True


## 6. Verify the saved evidence
Check linkage, evidence hashes, trial counts and equivalence. Reject a stale benchmark from another acquisition.

In [ ]:
import hashlib
assert BENCHMARK_COMPLETED
pointer = json.loads((WORKSPACE / "active_storage.json").read_text())
EVIDENCE = WORKSPACE / pointer["path"]
def digest(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()
assert pointer["evidence_sha256"] == digest(EVIDENCE / "evidence.json")
result = json.loads((EVIDENCE / "evidence.json").read_text())
current_source = json.loads((WORKSPACE / "active_source.json").read_text())
assert result["status"] == "PASS"
assert result["source_manifest_sha256"] == current_source["manifest_sha256"]
assert result["total_trials"] == 22*(WARMUPS+REPETITIONS)
assert result["measured_trials"] == 22*REPETITIONS
for filename, expected in result["files"].items():
    assert digest(EVIDENCE / filename) == expected, filename
print("Storage evidence verified:", EVIDENCE.name)
print("Measured trials:", result["measured_trials"])


## 7. Measured results
Use the actual tables below to justify storage choices. Compare flat versus partitioned layouts within a codec, Snappy versus ZSTD within a layout, and CSV against both. Consider query latency, size and whole-worker memory together.

In [ ]:
import csv
import html
from IPython.display import display, HTML, Image

def show_csv(filename, columns=None, where=None):
    with (EVIDENCE / filename).open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if where:
        rows = [r for r in rows if where(r)]
    columns = columns or list(rows[0])
    def fmt(s):
        try:
            return f"{float(s):.4f}" if "." in s or "e-" in s else s
        except ValueError:
            return s
    table = "<table><thead><tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in columns) + "</tr></thead><tbody>"
    table += "".join("<tr>" + "".join(f"<td>{html.escape(fmt(r[c]))}</td>" for c in columns) + "</tr>" for r in rows)
    display(HTML(table + "</tbody></table>"))

show_csv("sizes.csv")
show_csv("summary.csv", ["format", "workload", "n", "median_seconds", "iqr_seconds", "median_peak_rss_mib", "rows", "candidate_fragments", "total_fragments"], lambda r: r["workload"] != "csv_projection")
print("Supplementary CSV reader experiment")
show_csv("summary.csv", ["format", "n", "median_seconds", "iqr_seconds", "median_peak_rss_mib"], lambda r: r["workload"] == "csv_projection")
for f in ("storage_size.png", "latency.png", "memory.png"):
    display(Image(filename=str(EVIDENCE / f)))


## 8. Partition pruning verification and interpretation
The year and quarter queries should select four and one candidate files from the 170 partitioned files. The flat representation may still exploit row-group statistics. A lower candidate-file count is not itself a measured reduction in physical I/O.

In [ ]:
with (EVIDENCE / "summary.csv").open(newline="", encoding="utf-8") as f:
    summary = list(csv.DictReader(f))
for row in summary:
    if row["format"].startswith("partitioned"):
        expected = {"year_2020": 4, "quarter_2020q1": 1}.get(row["workload"], 170)
        assert int(row["candidate_fragments"]) == expected
        assert int(row["total_fragments"]) == 170
print("Partition candidate-file checks: PASS")


## 9. Handoff
Part 1 is complete only when source and benchmark gates pass. Save the executed notebook with its actual outputs. Part 2 reads the Bronze source; the benchmark replicas are temporary and do not enter the model. The report should discuss the measured trade-offs and distinguish local CPU experiments from distributed scaling evidence.